# Exercises 1 — Foundations (Student)

ใช้ฟังก์ชันหลักจาก Lab เพื่อฝึกคิดเรื่อง **User Profile, Transformation และ Metric** โดยไม่เขียน pipeline ใหม่ทั้งหมด

In [1]:
# ============================================================
# Part 0 — Environment & Dataset
# ============================================================
!pip -q install gradio gensim sentence-transformers transformers

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import math, random, warnings
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_auc_score, mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")

BASE = Path("/content/drive/MyDrive/Colab Notebooks")
DATA_ROOT = BASE / "Dataset_Chapter3_4"

if (DATA_ROOT / "data").exists():
    DATA_DIR = DATA_ROOT / "data"
else:
    DATA_DIR = DATA_ROOT

print("BASE      =", BASE)
print("DATA_ROOT =", DATA_ROOT)
print("DATA_DIR  =", DATA_DIR)

items = pd.read_csv(DATA_DIR / "metadata.csv")
ratings = pd.read_csv(DATA_DIR / "ratings.csv")

rename_items = {
    "movie_id":"item_id", "id":"item_id",
    "movie_title":"title", "name":"title",
    "genres":"genre", "description":"synopsis",
    "plot":"synopsis", "summary":"synopsis"
}
for old,new in rename_items.items():
    if old in items.columns and new not in items.columns:
        items = items.rename(columns={old:new})

rename_r = {"uid":"user_id","user":"user_id","iid":"item_id","movie_id":"item_id","score":"rating"}
for old,new in rename_r.items():
    if old in ratings.columns and new not in ratings.columns:
        ratings = ratings.rename(columns={old:new})

required_i = ["item_id","title","genre","synopsis"]
required_r = ["user_id","item_id","rating"]
assert all(c in items.columns for c in required_i), items.columns.tolist()
assert all(c in ratings.columns for c in required_r), ratings.columns.tolist()

items["item_id"] = items["item_id"].astype(str)
ratings["item_id"] = ratings["item_id"].astype(str)
ratings["user_id"] = ratings["user_id"].astype(str)
ratings["rating"] = pd.to_numeric(ratings["rating"])

items["genre"] = items["genre"].fillna("").astype(str)
items["synopsis"] = items["synopsis"].fillna("").astype(str)
items["text_content"] = (items["genre"] + " " + items["synopsis"]).str.strip()

ITEM_INDEX = {iid:i for i,iid in enumerate(items["item_id"])}
N_USERS = max(1, ratings["user_id"].nunique())
POPULARITY = ratings.groupby("item_id")["user_id"].nunique().to_dict()

print(f"Items={len(items)}, Users={ratings['user_id'].nunique()}, Ratings={len(ratings)}")
display(items[["item_id","title","genre","synopsis"]].head())


# ============================================================
# Core Functions
# Vector Transformation + 3 User Profiles + Ranking + Metrics
# ============================================================

LIKED_THRESHOLD = 4.0
DISLIKED_THRESHOLD = 2.0

def transform_vectors(X, mode="Raw+L2"):
    X = np.asarray(X, dtype=float)
    if mode == "Raw+L2":
        return normalize(X)
    if mode == "Center+L2":
        return normalize(X - X.mean(axis=0, keepdims=True))
    if mode == "Standard+L2":
        return normalize(StandardScaler().fit_transform(X))
    raise ValueError(mode)

def _unit(v):
    v=np.asarray(v,dtype=float)
    n=np.linalg.norm(v)
    return v if n==0 else v/n

def profile_positive_only(user_id, X, ratings_df):
    r=ratings_df[ratings_df["user_id"].astype(str)==str(user_id)]
    liked=r[r["rating"]>=LIKED_THRESHOLD]
    idx=[ITEM_INDEX[str(i)] for i in liked["item_id"] if str(i) in ITEM_INDEX]
    if not idx:
        idx=[ITEM_INDEX[str(i)] for i in r["item_id"] if str(i) in ITEM_INDEX]
    return _unit(X[idx].mean(axis=0)) if idx else np.zeros(X.shape[1])

def profile_rating_centered(user_id, X, ratings_df):
    r=ratings_df[ratings_df["user_id"].astype(str)==str(user_id)].copy()
    r=r[r["item_id"].astype(str).isin(ITEM_INDEX)]
    if r.empty: return np.zeros(X.shape[1])
    idx=np.array([ITEM_INDEX[str(i)] for i in r["item_id"]])
    w=r["rating"].astype(float).values - float(r["rating"].mean())
    denom=np.abs(w).sum()
    p=X[idx].mean(axis=0) if denom<=1e-12 else (X[idx]*w[:,None]).sum(axis=0)/denom
    return _unit(p)

def profile_rocchio(user_id, X, ratings_df, alpha=1.0, beta=1.0):
    r=ratings_df[ratings_df["user_id"].astype(str)==str(user_id)]
    liked=r[r["rating"]>=LIKED_THRESHOLD]
    disliked=r[r["rating"]<=DISLIKED_THRESHOLD]
    li=[ITEM_INDEX[str(i)] for i in liked["item_id"] if str(i) in ITEM_INDEX]
    di=[ITEM_INDEX[str(i)] for i in disliked["item_id"] if str(i) in ITEM_INDEX]
    pos=X[li].mean(axis=0) if li else np.zeros(X.shape[1])
    neg=X[di].mean(axis=0) if di else np.zeros(X.shape[1])
    p=alpha*pos-beta*neg
    if np.linalg.norm(p)==0:
        hi=[ITEM_INDEX[str(i)] for i in r["item_id"] if str(i) in ITEM_INDEX]
        p=X[hi].mean(axis=0) if hi else p
    return _unit(p)

PROFILE_METHODS = {
    "Positive-only": profile_positive_only,
    "Rating-centered": profile_rating_centered,
    "Rocchio": profile_rocchio,
}

def get_profile(user_id, X, ratings_df, method, alpha=1.0, beta=1.0):
    if method=="Rocchio":
        return profile_rocchio(user_id,X,ratings_df,alpha,beta)
    return PROFILE_METHODS[method](user_id,X,ratings_df)

def rank_user(user_id, X, ratings_df, method, k=None, candidates=None, alpha=1.0, beta=1.0):
    p=get_profile(user_id,X,ratings_df,method,alpha,beta)
    score=cosine_similarity(p.reshape(1,-1),X).ravel()
    seen=set(ratings_df.loc[ratings_df["user_id"].astype(str)==str(user_id),"item_id"].astype(str))
    if candidates is None:
        candidates=[iid for iid in items["item_id"] if iid not in seen]
    ranked=sorted(candidates,key=lambda iid:score[ITEM_INDEX[iid]],reverse=True)
    return ranked[:k] if k else ranked, score, p

# Accuracy / Ranking Metrics
def precision_at_k(recs, rel, k):
    return len(set(recs[:k]) & set(rel))/k if k else 0.0
def recall_at_k(recs, rel, k):
    rel=set(rel); return len(set(recs[:k])&rel)/len(rel) if rel else 0.0
def f1_at_k(recs, rel, k):
    p=precision_at_k(recs,rel,k); r=recall_at_k(recs,rel,k)
    return 0.0 if p+r==0 else 2*p*r/(p+r)
def hr_at_k(recs, rel, k):
    return float(bool(set(recs[:k])&set(rel)))
def reciprocal_rank(recs, rel):
    rel=set(rel)
    for rank,iid in enumerate(recs,1):
        if iid in rel: return 1.0/rank
    return 0.0
def ap_at_k(recs, rel, k):
    rel=set(rel); hits=0; s=0.0
    for i,iid in enumerate(recs[:k],1):
        if iid in rel:
            hits+=1; s+=hits/i
    return s/min(len(rel),k) if rel else 0.0
def ndcg_at_k(recs, rel, k):
    rel=set(rel)
    dcg=sum(1/math.log2(i+2) for i,iid in enumerate(recs[:k]) if iid in rel)
    idcg=sum(1/math.log2(i+2) for i in range(min(len(rel),k)))
    return dcg/idcg if idcg else 0.0

# Beyond-Accuracy Metrics
def coverage(all_recs):
    return len(set(all_recs))/len(items) if len(items) else 0.0

def gini_coefficient(all_recs):
    counts=pd.Series(all_recs).value_counts()
    x=np.array([counts.get(iid,0) for iid in items["item_id"]],dtype=float)
    if x.sum()==0: return 0.0
    x=np.sort(x); n=len(x)
    return float((2*np.sum(np.arange(1,n+1)*x)/(n*x.sum()))-(n+1)/n)

def ild(item_ids, X):
    idx=[ITEM_INDEX[i] for i in item_ids if i in ITEM_INDEX]
    if len(idx)<2: return 0.0
    D=1-cosine_similarity(X[idx])
    v=D[np.triu_indices_from(D,k=1)]
    return float(v.mean()) if len(v) else 0.0

def novelty(item_ids):
    vals=[]
    for iid in item_ids:
        pop=max(1,POPULARITY.get(iid,1))
        vals.append(-math.log2(pop/N_USERS))
    return float(np.mean(vals)) if vals else 0.0

def calibration_genre(user_id, rec_ids, ratings_df):
    hist=ratings_df[ratings_df["user_id"].astype(str)==str(user_id)]
    hist_genres=[]
    for iid in hist["item_id"].astype(str):
        if iid in ITEM_INDEX:
            hist_genres += [g.strip() for g in str(items.iloc[ITEM_INDEX[iid]]["genre"]).replace("|",",").split(",") if g.strip()]
    rec_genres=[]
    for iid in rec_ids:
        rec_genres += [g.strip() for g in str(items.iloc[ITEM_INDEX[iid]]["genre"]).replace("|",",").split(",") if g.strip()]
    if not hist_genres or not rec_genres: return np.nan
    cats=sorted(set(hist_genres)|set(rec_genres))
    hp=np.array([hist_genres.count(c) for c in cats],float); hp/=hp.sum()
    rp=np.array([rec_genres.count(c) for c in cats],float); rp/=rp.sum()
    return float(1-0.5*np.abs(hp-rp).sum())

def loo_folds(ratings_df, positive_threshold=4.0):
    folds=[]
    for uid,g in ratings_df.groupby("user_id"):
        pos=g[g["rating"]>=positive_threshold]
        if pos.empty: continue
        hold=pos.sort_values("rating").iloc[-1]
        folds.append((str(uid),ratings_df.drop(index=hold.name),str(hold["item_id"])))
    return folds

def evaluate_config(X, , k=3, alpha=1.0, beta=1.0, folds=None,
                    candidate_mode="Full Catalog", n_neg=20, seed=42):
    folds=folds or loo_folds(ratings)
    rng=np.random.default_rng(seed)
    rows=[]; global_recs=[]
    for uid,train,heldout in folds:
        seen=set(train.loc[train["user_id"].astype(str)==uid,"item_id"].astype(str))
        all_c=[iid for iid in items["item_id"] if iid not in seen]
        if heldout not in all_c: all_c.append(heldout)
        if candidate_mode=="Sampled Negatives":
            neg=[x for x in all_c if x!=heldout]
            sampled=list(rng.choice(neg,size=min(n_neg,len(neg)),replace=False)) if neg else []
            candidates=[heldout]+sampled
        else:
            candidates=all_c
        ranked,score,_=rank_user(uid,X,train,profile_method,candidates=candidates,alpha=alpha,beta=beta)
        topk=ranked[:k]; global_recs += topk
        y=np.array([1 if iid==heldout else 0 for iid in ranked])
        ys=np.array([score[ITEM_INDEX[iid]] for iid in ranked])
        auc=roc_auc_score(y,ys) if len(np.unique(y))==2 else np.nan
        rows.append({
            "user_id":uid,"heldout":heldout,
            f"Precision@{k}":precision_at_k(topk,[heldout],k),
            f"Recall@{k}":recall_at_k(topk,[heldout],k),
            f"F1@{k}":f1_at_k(topk,[heldout],k),
            f"HR@{k}":hr_at_k(topk,[heldout],k),
            "MRR":reciprocal_rank(ranked,[heldout]),
            f"MAP@{k}":ap_at_k(topk,[heldout],k),
            f"nDCG@{k}":ndcg_at_k(topk,[heldout],k),
            "AUC":auc,"ILD":ild(topk,X),"Novelty":novelty(topk),
            "Calibration":calibration_genre(uid,topk,train),
            "TopK":" | ".join(topk)
        })
    detail=pd.DataFrame(rows)
    summary=detail.select_dtypes(include=np.number).mean().to_dict() if len(detail) else {}
    summary["Coverage"]=coverage(global_recs)
    summary["Gini"]=gini_coefficient(global_recs)
    return pd.Series(summary), detail

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 46.6 MB/s eta 0:00:00
Mounted at /content/drive
BASE      = /content/drive/MyDrive/Colab Notebooks
DATA_ROOT = /content/drive/MyDrive/Colab Notebooks/Dataset_Chapter3_4
DATA_DIR  = /content/drive/MyDrive/Colab Notebooks/Dataset_Chapter3_4/data
Items=12, Users=8, Ratings=32


,item_id,title,genre,synopsis
0,i1,Toy Story 5,Animation|Family,A cowboy toy and a space ranger toy lead the t...
1,i2,Inside Out 2,Animation|Family,"Inside the mind of a teenage girl, five emotio..."
2,i3,The Wild Robot,Animation|Sci-Fi,A shipwrecked robot boots up on a wild island ...
3,i4,It Ends with Us,Drama|Romance,A florist falls in love with a charming surgeo...
4,i5,Wicked,Fantasy|Musical,A green-skinned woman with a gift for magic jo...


In [2]:
tfidf=TfidfVectorizer(lowercase=True,ngram_range=(1,2),min_df=1)
X_raw=tfidf.fit_transform(items["text_content"]).toarray()
VECTOR_SPACES={m:transform_vectors(X_raw,m) for m in ["Raw+L2","Center+L2","Standard+L2"]}

## ข้อ 1 — เปรียบเทียบ User Profile 3 วิธี
เลือก User 1 คน แล้วเรียก `get_profile()` ทั้ง 3 วิธี จากนั้นคำนวณ cosine ระหว่าง profile คู่ต่าง ๆ

In [5]:
USER_ID = 'u3'

# TODO 1: ใช้ X = VECTOR_SPACES["Raw+L2"]
# TODO 2: เรียก get_profile() สำหรับ Positive-only, Rating-centered, Rocchio
# TODO 3: สร้างตาราง cosine similarity 3x3

example_user=USER_ID
X=VECTOR_SPACES["Center+L2"]
rows=[]; profiles={}
for method in PROFILE_METHODS:
    p=get_profile(example_user,X,ratings,method)
    profiles[method]=p
    ranked,score,_=rank_user(example_user,X,ratings,method,k=3)
    rows.append({"Profile":method,
                 "Top-3":" | ".join(items.iloc[ITEM_INDEX[i]]["title"] for i in ranked)})
display(pd.DataFrame(rows))

names=list(profiles)
M=np.array([[cosine_similarity(profiles[a].reshape(1,-1),profiles[b].reshape(1,-1))[0,0]
             for b in names] for a in names])
display(pd.DataFrame(M,index=names,columns=names).round(4))

,Profile,Top-3
0,Positive-only,Furiosa: A Mad Max Saga | The Wild Robot | Con...
1,Rating-centered,The Wild Robot | Inside Out 2 | Toy Story 5
2,Rocchio,Toy Story 5 | The Wild Robot | Inside Out 2


,Positive-only,Rating-centered,Rocchio
Positive-only,1.0000,0.5780,0.5938
Rating-centered,0.5780,1.0000,0.9697
Rocchio,0.5938,0.9697,1.0000


## ข้อ 2 — Rocchio Sensitivity
เรียก `rank_user()` โดยปรับ `beta = 0.5, 1.0, 2.0` และบันทึก Top-3  
**คิดก่อน Run:** beta สูงขึ้นควรมีผลอย่างไรต่อ item ที่คล้าย disliked items?

In [7]:
# TODO: loop beta แล้วสร้าง DataFrame เปรียบเทียบ Top-3

example_user=sorted(ratings["user_id"].unique())[0]
X=VECTOR_SPACES["Raw+L2"]
rows=[]; profiles={}
for b in [0.5,1.0,2.0]:
    p=get_profile(example_user,X,ratings,"Rocchio")
    profiles[method]=p
    ranked,score,_=rank_user(example_user,X,ratings,"Rocchio",k=3, beta =b)
    rows.append({"beta":b,
                 "Top-3":" | ".join(items.iloc[ITEM_INDEX[i]]["title"] for i in ranked)})
display(pd.DataFrame(rows))



,beta,Top-3
0,0.5,Dune: Part Two | Wicked | Mickey 17
1,1.0,Dune: Part Two | Mickey 17 | Conclave
2,2.0,Dune: Part Two | Furiosa: A Mad Max Saga | Con...


## ข้อ 3 — Transformation
ใช้ `Rating-centered` คงที่ แต่เปลี่ยน 3 transformations แล้วเรียก `evaluate_config()`  
รายงาน HR@3, nDCG@3, Coverage, ILD, Novelty

In [10]:
# TODO

folds=loo_folds(ratings)
rows=[]
for tname,X in VECTOR_SPACES.items():
        s,_=evaluate_config(X,'Rating-centered',k=3,folds=folds,candidate_mode="Full Catalog")
        rows.append({"Transformation":tname,"Profile":'Rating-centered',**s.to_dict()})
lab1_results=pd.DataFrame(rows)
display(lab1_results.round(4))


,Transformation,Profile,Precision@3,Recall@3,F1@3,HR@3,MRR,MAP@3,nDCG@3,AUC,ILD,Novelty,Calibration,Coverage,Gini
0,Raw+L2,Rating-centered,0.2083,0.625,0.3125,0.625,0.6958,0.6250,0.6250,0.7969,0.9360,1.5236,0.4479,0.9167,0.2986
1,Center+L2,Rating-centered,0.2083,0.625,0.3125,0.625,0.6958,0.6250,0.6250,0.7969,1.0895,1.5236,0.4479,0.9167,0.2986
2,Standard+L2,Rating-centered,0.2083,0.625,0.3125,0.625,0.6396,0.5625,0.5789,0.7969,1.0928,1.6140,0.3854,1.0000,0.1875


## ข้อ 4 — Interpretation
อธิบายว่าเหตุใด configuration ที่ nDCG@3 สูงสุดอาจไม่ใช่ configuration ที่ ILD หรือ Novelty สูงสุด